# LSM — Entrenamiento en Colab T4
**Antes de correr:** Runtime → Change runtime type → **T4 GPU**

Corre todas las celdas en orden (Ctrl+F9).

In [ ]:
# ── Celda 1: Montar Drive y descomprimir ZIP ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, glob

ZIP_PATH   = '/content/drive/MyDrive/LSM/LSM_Colab.zip'
REPO_LOCAL = '/content/LSM'
DRIVE_MODELS = '/content/drive/MyDrive/LSM/models'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'No encontre el ZIP en {ZIP_PATH}')

print(f'ZIP encontrado: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)')

# Descomprimir al disco local de Colab (mucho mas rapido que leer de Drive)
if os.path.exists(REPO_LOCAL):
    import shutil; shutil.rmtree(REPO_LOCAL)

os.makedirs(REPO_LOCAL, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(REPO_LOCAL)

# El ZIP puede haber creado una subcarpeta con el mismo nombre — nos movemos adentro
contents = os.listdir(REPO_LOCAL)
if len(contents) == 1 and os.path.isdir(os.path.join(REPO_LOCAL, contents[0])):
    REPO_LOCAL = os.path.join(REPO_LOCAL, contents[0])

os.chdir(REPO_LOCAL)
print(f'Directorio de trabajo: {os.getcwd()}')
npy = glob.glob('data/**/*.npy', recursive=True)
print(f'Archivos .npy encontrados: {len(npy)}')

In [ ]:
# ── Celda 2: Instalar dependencias ───────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy', 'scikit-learn', 'joblib'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
# ── Celda 3: Entrenar modelo UNIFICADO (letras + palabras, un solo LSTM) ──
# Un modelo que reconoce todo: j, k, q, x, z, ñ + 250 palabras en español
from train_models import train_unified
train_unified()

In [ ]:
# ── Celda 4: Verificar resultados ────────────────────────────────────────
import json, os

jf = 'models/unified_classes.json'
pf = 'models/unified_model.pth'
classes = json.load(open(jf, encoding='utf-8'))
mb = os.path.getsize(pf) / 1e6
print(f'Modelo unificado: {len(classes)} clases, {mb:.1f} MB')
print('Letras:', [c for c in classes if len(c) == 1])
print('Palabras (muestra):', [c for c in classes if len(c) > 1][:10], '...')

In [ ]:
# ── Celda 5: Guardar modelo en Drive ─────────────────────────────────────
import shutil, os

os.makedirs(DRIVE_MODELS, exist_ok=True)

for f in ['unified_model.pth', 'unified_classes.json']:
    shutil.copy2(f'models/{f}', f'{DRIVE_MODELS}/{f}')
    print(f'Guardado: {DRIVE_MODELS}/{f}')

print(f'\nListo. Descarga los 2 archivos de Drive: {DRIVE_MODELS}/')

## Siguiente paso
Descarga los 4 archivos desde Drive y ponlos en:
```
C:\Users\capa_\OneDrive\Desktop\LSM\models\
```
- `dynamic_letters_model.pth`
- `dynamic_letters_classes.json`
- `words_model.pth`
- `words_classes.json`